In [1]:
import pandas as pd
from io import StringIO
import requests
from bs4 import BeautifulSoup
from IPython.core.display import HTML
import re

## Webscraping


In this exercise, you'll practice using BeautifulSoup to parse the content of a web page. The page that you'll be scraping, https://realpython.github.io/fake-jobs/, contains job listings. Your job is to extract the data on each job and convert into a pandas DataFrame.

#### 1. Start by performing a GET request on the url above and convert the response into a BeautifulSoup object.  

In [5]:
URL = 'https://realpython.github.io/fake-jobs/'

headers = {
    "User-Agent": "PretendJobsHunter"
}

response = requests.get(URL, headers=headers)

In [6]:
mixture = BeautifulSoup(response.text)
mixture.head()

[<meta charset="utf-8"/>,
 <meta content="width=device-width, initial-scale=1" name="viewport"/>,
 <title>Fake Python</title>,
 <link href="https://cdn.jsdelivr.net/npm/bulma@0.9.2/css/bulma.min.css" rel="stylesheet"/>]

In [7]:
print(mixture.prettify())

<!DOCTYPE html>
<html>
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width, initial-scale=1" name="viewport"/>
  <title>
   Fake Python
  </title>
  <link href="https://cdn.jsdelivr.net/npm/bulma@0.9.2/css/bulma.min.css" rel="stylesheet"/>
 </head>
 <body>
  <section class="section">
   <div class="container mb-5">
    <h1 class="title is-1">
     Fake Python
    </h1>
    <p class="subtitle is-3">
     Fake Jobs for Your Web Scraping Journey
    </p>
   </div>
   <div class="container">
    <div class="columns is-multiline" id="ResultsContainer">
     <div class="column is-half">
      <div class="card">
       <div class="card-content">
        <div class="media">
         <div class="media-left">
          <figure class="image is-48x48">
           <img alt="Real Python Logo" src="https://files.realpython.com/media/real-python-logo-thumbnail.7f0db70c2ed2.jpg?__no_cf_polish=1"/>
          </figure>
         </div>
         <div class="media-content">
          <h2 c

#### 1a. Use the .find method to find the tag containing the first job title ("Senior Python Developer"). Hint: can you find a tag type and/or a class that could be helpful for extracting this information? Extract the text from this title.  

In [9]:
mixture.findAll('h2')[0]#[:5]#.text

<h2 class="title is-5">Senior Python Developer</h2>

In [10]:
mixture.find('h2').text

'Senior Python Developer'

#### 1b. Now, use what you did for the first title, but extract the job title for all jobs on this page. Store the results in a list.  

In [12]:
job_titles = mixture.findAll('h2')#[:5]#.text

#### 1c. Finally, extract the companies, locations, and posting dates for each job. For example, the first job has a company of "Payne, Roberts and Davis", a location of "Stewartbury, AA", and a posting date of "2021-04-08". Ensure that the text that you extract is clean, meaning no extra spaces or other characters at the beginning or end.  

In [14]:
job_titles = mixture.findAll('h2')#[:5]#.text
job_titles[0].text
#job_titles

'Senior Python Developer'

In [15]:
list_job_titles = [i.text for i in job_titles]
list_job_titles[:2]
#HTML(list_job_titles)

['Senior Python Developer', 'Energy engineer']

In [16]:
company_names = mixture.findAll('h3')#[:5]#.text
company_names[:5]
list_company_names = [i.text for i in company_names]
list_company_names[:2]

['Payne, Roberts and Davis', 'Vasquez-Davidson']

In [17]:
posting_dates = mixture.findAll('time')
list_posting_dates = [i.text for i in posting_dates]
list_posting_dates[:2]

['2021-04-08', '2021-04-08']

In [18]:
#locations = 
locs = mixture.find('p', attrs={'class' : 'location'}).text
#mixture.find('p', attrs={'class' : 'location'})
#clean_locations = BeautifulSoup(locs, "lxml").text
#locations
#list_locations = [i.text for i in locations]
#list_locations[:2]
locs

'\n        Stewartbury, AA\n      '

In [19]:
location01 = ''.join(locs.split())
location01

'Stewartbury,AA'

In [20]:
re.sub(r'(?<=[.,])(?=[^\s])', r' ', location01)

'Stewartbury, AA'

In [21]:
#locations = 
locations = mixture.findAll('p', attrs={'class' : 'location'})#[:2]
#mixture.find('p', attrs={'class' : 'location'})
#clean_locations = BeautifulSoup(locs, "lxml").text
#locations
#list_locations = [i.text for i in locations]
#list_locations[:2]
#location01 = ''.join(locs.split())
strange_list_locations = [i.text for i in locations]

In [22]:
strange_list_locations[:2]

['\n        Stewartbury, AA\n      ', '\n        Christopherville, AA\n      ']

In [23]:
funky_loc_list = [''.join(i.split()) for i in strange_list_locations]

In [24]:
job_locations = [re.sub(r'(?<=[.,])(?=[^\s])', r' ', x) for x in funky_loc_list]

In [25]:
job_locations[:1]

['Stewartbury, AA']

#### 1d. Take the lists that you have created and combine them into a pandas DataFrame. 

In [27]:
#[x.get('') for x in job_titles]

In [28]:
pretend_jobs_table = pd.DataFrame(
    {'job_titles': list_job_titles,
     'company': list_company_names,
     'locations': job_locations,
     'posting_dates': list_posting_dates
    })

In [29]:
pretend_jobs_table

,job_titles,company,locations,posting_dates
0,Senior Python Developer,"Payne, Roberts and Davis","Stewartbury, AA",2021-04-08
1,Energy engineer,Vasquez-Davidson,"Christopherville, AA",2021-04-08
2,Legal executive,"Jackson, Chambers and Levy","PortEricaburgh, AA",2021-04-08
3,Fitness centre manager,Savage-Bradley,"EastSeanview, AP",2021-04-08
4,Product manager,Ramirez Inc,"NorthJamieview, AP",2021-04-08
...,...,...,...,...
95,Museum/gallery exhibitions officer,"Nguyen, Yoder and Petty","LakeAbigail, AE",2021-04-08
96,"Radiographer, diagnostic",Holder LLC,"Jacobshire, AP",2021-04-08
97,Database administrator,Yates-Ferguson,"PortSusan, AE",2021-04-08
98,Furniture designer,Ortega-Lawrence,"NorthTiffany, AA",2021-04-08


#### 2a. Next, add a column that contains the url for the "Apply" button. Try this in two ways.  a. First, use the BeautifulSoup find_all method to extract the urls.  

In [31]:
#mixture.findAll('footer')
#huh = 
mixture.find('footer')
#.text


<footer class="card-footer">
<a class="card-footer-item" href="https://www.realpython.com" target="_blank">Learn</a>
<a class="card-footer-item" href="https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html" target="_blank">Apply</a>
</footer>

In [32]:
mixture.findAll('a')[1::2][0]

<a class="card-footer-item" href="https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html" target="_blank">Apply</a>

In [33]:
mixture.find('footer', attrs={'a href'})


In [34]:
mixture.find('a')

<a class="card-footer-item" href="https://www.realpython.com" target="_blank">Learn</a>

In [35]:
mixture.findAll('a', attrs={'href': "http://"})

[]

In [36]:
whole_links_list = [link.get('href') for link in mixture.findAll('a')]
whole_links_list[1]

'https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html'

In [37]:
bare_links_list = [x for x in whole_links_list]

In [38]:
apply_links = bare_links_list[1::2]

In [39]:
pretend_jobs_table_links_included = pd.DataFrame(
    {'job_titles': list_job_titles,
     'company': list_company_names,
     'locations': job_locations,
     'posting_dates': list_posting_dates, 
     'apply_link' : apply_links
    })
pretend_jobs_table_links_included

,job_titles,company,locations,posting_dates,apply_link
0,Senior Python Developer,"Payne, Roberts and Davis","Stewartbury, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/se...
1,Energy engineer,Vasquez-Davidson,"Christopherville, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/en...
2,Legal executive,"Jackson, Chambers and Levy","PortEricaburgh, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/le...
3,Fitness centre manager,Savage-Bradley,"EastSeanview, AP",2021-04-08,https://realpython.github.io/fake-jobs/jobs/fi...
4,Product manager,Ramirez Inc,"NorthJamieview, AP",2021-04-08,https://realpython.github.io/fake-jobs/jobs/pr...
...,...,...,...,...,...
95,Museum/gallery exhibitions officer,"Nguyen, Yoder and Petty","LakeAbigail, AE",2021-04-08,https://realpython.github.io/fake-jobs/jobs/mu...
96,"Radiographer, diagnostic",Holder LLC,"Jacobshire, AP",2021-04-08,https://realpython.github.io/fake-jobs/jobs/ra...
97,Database administrator,Yates-Ferguson,"PortSusan, AE",2021-04-08,https://realpython.github.io/fake-jobs/jobs/da...
98,Furniture designer,Ortega-Lawrence,"NorthTiffany, AA",2021-04-08,https://realpython.github.io/fake-jobs/jobs/fu...


In [40]:
#[s for s in list if sub in s]

#### 2b. Next, add a column that contains the url for the "Apply" button. Try this in two ways. b. Next, get those same urls in a different way. Examine the urls and see if you can spot the pattern of how they are constructed. Then, build the url using the elements you have already extracted. Ensure that the urls that you created match those that you extracted using BeautifulSoup. Warning: You will need to do some string cleaning and prep in constructing the urls this way. For example, look carefully at the urls for the "Software Engineer (Python)" job and the "Scientist, research (maths)" job. 

In [42]:
list_job_titles[10]

'Software Engineer (Python)'

In [43]:
re.sub(r'\([^)]*\)', '', list_job_titles[10])

'Software Engineer '

In [44]:
#[re.sub(r'(?<=[.,])(?=[^\s])', r' ', x) for x in list_job_titles]

In [45]:
re.sub(r'\([^()]*\)', '', list_job_titles[10])

'Software Engineer '

In [46]:
#re.sub(r"\()*", '', list_job_titles[10])

In [47]:
no_slash = [re.sub(r"\(?\/", " ", i) for i in list_job_titles]
no_slash[:5]

['Senior Python Developer',
 'Energy engineer',
 'Legal executive',
 'Fitness centre manager',
 'Product manager']

In [48]:
no_one_parentheses  = [re.sub(r"\)", "", i) for i in no_slash]
no_one_parentheses[:5]

['Senior Python Developer',
 'Energy engineer',
 'Legal executive',
 'Fitness centre manager',
 'Product manager']

In [49]:
no_last_parentheses  = [re.sub(r"\(", "", i) for i in no_one_parentheses]
no_last_parentheses[:5]

['Senior Python Developer',
 'Energy engineer',
 'Legal executive',
 'Fitness centre manager',
 'Product manager']

In [50]:
#well = []
#for i in no_last_parentheses:
#    um = i.replace(' ', '') 
#    well.append(um)
#well


In [51]:
ending_dash = [i + "-" for i in no_last_parentheses]
ending_dash[:5]

['Senior Python Developer-',
 'Energy engineer-',
 'Legal executive-',
 'Fitness centre manager-',
 'Product manager-']

In [52]:
no_commas = [i.replace(", ", '-')for i in ending_dash]
no_commas[:5]

['Senior Python Developer-',
 'Energy engineer-',
 'Legal executive-',
 'Fitness centre manager-',
 'Product manager-']

In [53]:
no_spaces = [i.replace(" ", '-')for i in no_commas]
no_spaces

['Senior-Python-Developer-',
 'Energy-engineer-',
 'Legal-executive-',
 'Fitness-centre-manager-',
 'Product-manager-',
 'Medical-technical-officer-',
 'Physiological-scientist-',
 'Textile-designer-',
 'Television-floor-manager-',
 'Waste-management-officer-',
 'Software-Engineer-Python-',
 'Interpreter-',
 'Architect-',
 'Meteorologist-',
 'Audiological-scientist-',
 'English-as-a-second-language-teacher-',
 'Surgeon-',
 'Equities-trader-',
 'Newspaper-journalist-',
 'Materials-engineer-',
 'Python-Programmer-Entry-Level-',
 'Product-process-development-scientist-',
 'Scientist-research-maths-',
 'Ecologist-',
 'Materials-engineer-',
 'Historic-buildings-inspector-conservation-officer-',
 'Data-scientist-',
 'Psychiatrist-',
 'Structural-engineer-',
 'Immigration-officer-',
 'Python-Programmer-Entry-Level-',
 'Neurosurgeon-',
 'Broadcast-engineer-',
 'Make-',
 'Nurse-adult-',
 'Air-broker-',
 'Editor-film-video-',
 'Production-assistant-radio-',
 'Engineer-communications-',
 'Sales-e

In [54]:
no_spaces[0] = no_spaces[0].ljust(len(no_spaces[0])+1, '0')

In [55]:
no_spaces[:5]

['Senior-Python-Developer-0',
 'Energy-engineer-',
 'Legal-executive-',
 'Fitness-centre-manager-',
 'Product-manager-']

In [56]:
b = [no_spaces[0] + str(i) for i in range(51)]
b[-5:]

['Senior-Python-Developer-046',
 'Senior-Python-Developer-047',
 'Senior-Python-Developer-048',
 'Senior-Python-Developer-049',
 'Senior-Python-Developer-050']

In [57]:
def add_numbers(df) :
    re.sub(r'[0-9]+$',
             lambda x: f"{str(int(x.group())+1).zfill(len(x.group()))}", 
             df)

In [58]:
#add_numbers(no_spaces)

In [59]:
#def increment_string(strng):
#    return re.sub(r'[0-9]+$', lambda x: f"{str(int(x.group())+1).zfill(len(x.group()))}", strng)

In [60]:
#no_spaces

#print("The original string is : " + str(no_spaces[0]))
#[re.sub(r'[0-9]+$', lambda word: f"{str(int(word.group())+1).zfill(len(word.group()))}", i) for i in no_spaces]
#    res = 
#print("Incremented numeric String: " + str(res))
#res

In [61]:
#no_spaces[0] = "Senior-Python-Developer-0"
#print("The original string is : " + str(this))
#res = re.sub(r'[0-9]+$',
#             lambda x: f"{str(int(x.group())+1).zfill(len(x.group()))}", 
#             this)
#print("Incremented numeric String: " + str(res))

In [62]:
#[i + "-" for i in dashes_in_middle]

In [63]:
um = re.sub(r"\(?\)", "", list_job_titles[10])
um

'Software Engineer (Python'

In [64]:
#um = re.sub(r"(\({1})?(\){1})", "", list_job_titles[10])
#um

In [65]:
re.sub(r"\)", "", um)

'Software Engineer (Python'

In [66]:
pretend_jobs_table['job_titles'].unique()

array(['Senior Python Developer', 'Energy engineer', 'Legal executive',
       'Fitness centre manager', 'Product manager',
       'Medical technical officer', 'Physiological scientist',
       'Textile designer', 'Television floor manager',
       'Waste management officer', 'Software Engineer (Python)',
       'Interpreter', 'Architect', 'Meteorologist',
       'Audiological scientist', 'English as a second language teacher',
       'Surgeon', 'Equities trader', 'Newspaper journalist',
       'Materials engineer', 'Python Programmer (Entry-Level)',
       'Product/process development scientist',
       'Scientist, research (maths)', 'Ecologist',
       'Historic buildings inspector/conservation officer',
       'Data scientist', 'Psychiatrist', 'Structural engineer',
       'Immigration officer', 'Neurosurgeon', 'Broadcast engineer',
       'Make', 'Nurse, adult', 'Air broker', 'Editor, film/video',
       'Production assistant, radio', 'Engineer, communications',
       'Sales execu

In [67]:
pretend_jobs_table

,job_titles,company,locations,posting_dates
0,Senior Python Developer,"Payne, Roberts and Davis","Stewartbury, AA",2021-04-08
1,Energy engineer,Vasquez-Davidson,"Christopherville, AA",2021-04-08
2,Legal executive,"Jackson, Chambers and Levy","PortEricaburgh, AA",2021-04-08
3,Fitness centre manager,Savage-Bradley,"EastSeanview, AP",2021-04-08
4,Product manager,Ramirez Inc,"NorthJamieview, AP",2021-04-08
...,...,...,...,...
95,Museum/gallery exhibitions officer,"Nguyen, Yoder and Petty","LakeAbigail, AE",2021-04-08
96,"Radiographer, diagnostic",Holder LLC,"Jacobshire, AP",2021-04-08
97,Database administrator,Yates-Ferguson,"PortSusan, AE",2021-04-08
98,Furniture designer,Ortega-Lawrence,"NorthTiffany, AA",2021-04-08


In [68]:
pretend_jobs_table_links_included['apply_link'].unique()

array(['https://realpython.github.io/fake-jobs/jobs/senior-python-developer-0.html',
       'https://realpython.github.io/fake-jobs/jobs/energy-engineer-1.html',
       'https://realpython.github.io/fake-jobs/jobs/legal-executive-2.html',
       'https://realpython.github.io/fake-jobs/jobs/fitness-centre-manager-3.html',
       'https://realpython.github.io/fake-jobs/jobs/product-manager-4.html',
       'https://realpython.github.io/fake-jobs/jobs/medical-technical-officer-5.html',
       'https://realpython.github.io/fake-jobs/jobs/physiological-scientist-6.html',
       'https://realpython.github.io/fake-jobs/jobs/textile-designer-7.html',
       'https://realpython.github.io/fake-jobs/jobs/television-floor-manager-8.html',
       'https://realpython.github.io/fake-jobs/jobs/waste-management-officer-9.html',
       'https://realpython.github.io/fake-jobs/jobs/software-engineer-python-10.html',
       'https://realpython.github.io/fake-jobs/jobs/interpreter-11.html',
       'https://r

In [69]:
#HTML(job_titles)

In [70]:
h2_header = mixture.findAll('h2')
first_job_title = h2_header[10]
first_job_title.text
#first_job_title#.get()

'Software Engineer (Python)'

In [71]:
h2_header = mixture.findAll('h2')
first_job_title = h2_header[0]
first_job_title.text#.get('class')

'Senior Python Developer'

In [72]:
h2_header = mixture.findAll('h2')
first_job_title = h2_header[0]
first_job_title.get('title')

In [73]:
mixture.find('p', attrs={'class' : 'location'})

<p class="location">
        Stewartbury, AA
      </p>

In [74]:
#[x.get('h2') for x in job_titles]
#titles_list

In [75]:
table_titles = str(mixture.find('h2', attrs={'class' : 'title'}))

HTML(table_titles)

In [76]:
#table_titles = (mixture.findAll('h2'))
#
#HTML(table_titles)

In [77]:
#HTML(job_titles)

In [78]:
mixture.findAll('class')#.text

[]

In [79]:
mixture.find('h2').text

'Senior Python Developer'

In [80]:
mixture.find('div').text

'\n\n        Fake Python\n      \n\n        Fake Jobs for Your Web Scraping Journey\n      \n'

In [81]:
job_titles = mixture.findAll('h2')#[:5]#.text

In [82]:
company_names = mixture.findAll('h3')#[:5]#.text

In [83]:
posting_dates = mixture.findAll('time')

In [84]:
#mixture.find('table', attrs={'class' : 'wikitable'})